In [9]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [10]:

# from langchain_community.vectorstores import FAISS
# #from langchain_ollama import OllamaEmbeddings
# from langchain_ollama import OllamaEmbeddings

# embeddings = OllamaEmbeddings(model="bge-m3")

# DB_INDEX = "LANGCHAIN_DB_INDEX"
# langgraph_db = FAISS.load_local(
#     DB_INDEX, embeddings=embeddings, allow_dangerous_deserialization=True,
# )

# print(langgraph_db.similarity_search("self-rag")[0].page_content)

In [11]:
# from retriever import FAISSRetrieverFactory

# query = "self-rag"

# fa = FAISSRetrieverFactory()

# fa_retriever = fa.retriever(index_path="LANGCHAIN_DB_INDEX", fetch_k=3)
# # Use the invoke() method to get relevant documents based on the query
# retrieved_docs = fa_retriever.invoke(query)

# print(retrieved_docs)

In [12]:
import sys, os, nest_asyncio, asyncio
from pathlib import Path
from mcp.client.stdio import stdio_client, StdioServerParameters
from mcp import ClientSession
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

nest_asyncio.apply()
os.environ.setdefault("PYTHONUNBUFFERED", "1")

async def run_mcp_client_stdio_in_notebook(
    server_file: str = "mcp_rag_stdio_qdrant.py",
    user_prompt: str = "DNA 서열 예측",
):
    server_path = Path(server_file).resolve()
    errlog_path = server_path.parent / "mcp_stderr.log"
    if not server_path.exists():
        raise FileNotFoundError(f"서버 스크립트 없음: {server_path}")

    if sys.platform == "win32":
        try: asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        except: pass

    server = StdioServerParameters(
        command=sys.executable,
        args=["-u", str(server_path)],
        cwd=str(server_path.parent),
        env=os.environ.copy(),
    )

    try:
        with open(errlog_path, "wb") as err:
            async with stdio_client(server, errlog=err) as (read, write):
                async with ClientSession(read, write) as session:
                    await session.initialize()
                    tools = await load_mcp_tools(session)
                    agent = create_react_agent(ChatOpenAI(model="gpt-4.1-mini", temperature=0), tools)
                    inputs = {"messages": [HumanMessage(content=user_prompt, name="user")]}
                    return [event async for event in agent.astream(inputs)]
    except Exception as e:
        tail = ""
        try:
            if errlog_path.exists():
                data = errlog_path.read_bytes()
                tail = (data[-4096:] if len(data) > 4096 else data).decode("utf-8", "ignore")
        except: pass
        raise RuntimeError(f"STDIO MCP 실행 실패: {e}\n\n=== server stderr (tail) ===\n{tail}") from e


In [13]:
# === mcp_stdio_helper.py (또는 노트북 첫 셀) ===
import sys, os, asyncio
from pathlib import Path
from contextlib import asynccontextmanager

# 주피터 환경 권장: nest_asyncio
try:
    import nest_asyncio
    nest_asyncio.apply()
except Exception:
    pass

os.environ.setdefault("PYTHONUNBUFFERED", "1")  # 서브프로세스 버퍼링 방지

from mcp.client.stdio import stdio_client, StdioServerParameters
from mcp import ClientSession
from langchain_mcp_adapters.tools import load_mcp_tools

def _win_proactor_policy():
    if sys.platform == "win32":
        try:
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        except Exception:
            pass

@asynccontextmanager
async def start_stdio_mcp(
    server_file: str,
    *,
    python: str = sys.executable,
    cwd: str | None = None,
    env: dict | None = None,
    errlog_path: str | None = "mcp_stderr.log",
):
    """
    STDIO MCP 서버를 서브프로세스로 올리고 ClientSession과 변환된 LangChain tools를 반환합니다.
    사용 예:
        async with start_stdio_mcp("server_stdio.py") as (session, tools):
            ...
    """
    _win_proactor_policy()

    server_path = Path(server_file).resolve()
    if not server_path.exists():
        raise FileNotFoundError(f"서버 스크립트가 없습니다: {server_path}")

    params = StdioServerParameters(
        command=python,
        args=["-u", str(server_path)],            # -u: unbuffered
        cwd=str(cwd or server_path.parent),
        env=(env or os.environ).copy(),
    )

    err_fp = None
    try:
        if errlog_path:
            err_fp = open(errlog_path, "wb")
        async with stdio_client(params, errlog=err_fp) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await load_mcp_tools(session)
                try:
                    yield session, tools
                finally:
                    # 세션 종료는 with 블록 해제 시 자동
                    pass
    except Exception as e:
        # 진단을 돕기 위해 stderr tail 표시
        tail = ""
        try:
            if errlog_path and Path(errlog_path).exists():
                data = Path(errlog_path).read_bytes()
                tail = (data[-4096:] if len(data) > 4096 else data).decode("utf-8", "ignore")
        except Exception:
            pass
        raise RuntimeError(f"STDIO MCP 연결 실패: {e}\n\n=== server stderr (tail) ===\n{tail}") from e
    finally:
        if err_fp:
            err_fp.close()


In [15]:
from langgraph.prebuilt import create_react_agent
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# 1) 서버 올리고 도구 받기
async with start_stdio_mcp("mcp_rag_stdio_faiss.py") as (session, tools):
    # 2) 에이전트 구성 & 실행ㅁ
    model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
    agent = create_agent(model, tools)

    inputs = {"messages": [HumanMessage(content="adaptive-rag", name="user")]}
    async for event in agent.astream(inputs):
        print(event)

    # # 한 번에 결과만:
    # result = await agent.ainvoke(inputs)
    # print(result)


{'model': {'messages': [AIMessage(content='Adaptive-RAG (Retrieval-Augmented Generation) is a method that combines retrieval-based techniques with generative models to improve the quality and relevance of generated responses. It adaptively retrieves relevant documents or information from a knowledge base and uses that context to generate more accurate and context-aware answers.\n\nIf you want, I can provide more detailed information about Adaptive-RAG, its architecture, use cases, or how it compares to other retrieval-augmented generation methods. Would you like me to do that?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 80, 'total_tokens': 180, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14',

In [6]:
# agent에서 tool로 안넘어가... 시간이 계속 지나는데도 말야
# 일단 sse로 해보니까 qdrant는 문제가 없어
# 그리고, mcp_server_local(stdio).py.는 문제없이 잘 실행이 돼

In [7]:
from langgraph.prebuilt import create_react_agent
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
async with start_stdio_mcp("mcp_rag_stdio_qdrant.py") as (session, tools):
    model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
    agent = create_agent(model, tools)
    
    inputs = {"messages": [HumanMessage(content="DNA 서열 예측", name="user")]}
    async for event in agent.astream(inputs):
        print(event)

{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 107, 'total_tokens': 124, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_4c2851f862', 'id': 'chatcmpl-CYQ6jy9twIHkrdJhdwGvYH91gRKF7', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--ee886ceb-2941-4863-a8ec-2a98eef741c9-0', tool_calls=[{'name': 'retrieve', 'args': {'query': 'DNA 서열 예측'}, 'id': 'call_wNm8MEhr9okhFYIkuihia7D6', 'type': 'tool_call'}], usage_metadata={'input_tokens': 107, 'output_tokens': 17, 'total_tokens': 124, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]}}
{'to

In [8]:
----

SyntaxError: invalid syntax (2133496677.py, line 1)

{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 80, 'total_tokens': 95, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_4c2851f862', 'id': 'chatcmpl-CYP9tTPv4zBXeA8bjFaTzgAWx4UuF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--306e47f5-77c0-4d61-9888-6964a2059453-0', tool_calls=[{'name': 'retrieve', 'args': {'query': 'adaptive-rag'}, 'id': 'call_tgRh97Fiy8SDSvW0qipI0hGg', 'type': 'tool_call'}], usage_metadata={'input_tokens': 80, 'output_tokens': 15, 'total_tokens': 95, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]}}


In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
async with start_stdio_mcp("mcp_rag_stdio_faiss.py.py") as (session, tools):
    model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
    agent = create_agent(model, tools)
    
    inputs = {"messages": [HumanMessage(content="adaptive-rag", name="user")]}
    async for event in agent.astream(inputs):
        print(event)

{'model': {'messages': [AIMessage(content='Adaptive-RAG (Retrieval-Augmented Generation) is a method in natural language processing that combines retrieval-based techniques with generative models to improve the quality and relevance of generated responses. It adaptively retrieves relevant documents or information from a large corpus and uses that information to guide the generation process, making the output more accurate and contextually appropriate.\n\nIf you want, I can provide more detailed information or examples about Adaptive-RAG. Would you like me to do that?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 80, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fi

In [ ]:
# from langgraph.prebuilt import create_react_agent
# from langchain.agents import create_agent
# from langchain_openai import ChatOpenAI
# from langchain_core.messages import HumanMessage
# # # 같은 디렉터리에 server_stdio.py 가 있어야 합니다.
# # server_stdio.py 안에서는 반드시 로그 print 를 stderr 로 보내세요:
# # print("MCP running...", file=sys.stderr)
# results = await run_mcp_client_stdio_in_notebook(
#     server_file="mcp_rag_stdio.py",
#     user_prompt="DNA서열예측",
# )

# # for r in results:
# #     print(r)


In [ ]:
-

SyntaxError: invalid syntax (476313318.py, line 1)

In [ ]:
# -----------------------------------------------------

In [ ]:
import sys, os, nest_asyncio, asyncio
from pathlib import Path
from mcp.client.stdio import stdio_client, StdioServerParameters
from mcp import ClientSession
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

nest_asyncio.apply()
os.environ.setdefault("PYTHONUNBUFFERED", "1")

async def run_mcp_client_stdio_in_notebook(
    server_file: str = "mcp_rag_stdio.py",

    user_prompt: str = "dna서열예측",
):
    server_path = Path(server_file).resolve()
    errlog_path = server_path.parent / "mcp_stderr.log"
    if not server_path.exists():
        raise FileNotFoundError(f"서버 스크립트 없음: {server_path}")

    if sys.platform == "win32":
        try: asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        except: pass

    server = StdioServerParameters(
        command=sys.executable,
        args=["-u", str(server_path)],
        cwd=str(server_path.parent),
        env=os.environ.copy(),
    )

    try:
        with open(errlog_path, "wb") as err:
            async with stdio_client(server, errlog=err) as (read, write):
                async with ClientSession(read, write) as session:
                    await session.initialize()
                    tools = await load_mcp_tools(session)
                    agent = create_react_agent(ChatOpenAI(model="gpt-4.1-mini", temperature=0), tools)
                    inputs = {"messages": [HumanMessage(content=user_prompt, name="user")]}
                    return [event async for event in agent.astream(inputs)]
    except Exception as e:
        tail = ""
        try:
            if errlog_path.exists():
                data = errlog_path.read_bytes()
                tail = (data[-4096:] if len(data) > 4096 else data).decode("utf-8", "ignore")
        except: pass
        raise RuntimeError(f"STDIO MCP 실행 실패: {e}\n\n=== server stderr (tail) ===\n{tail}") from e


In [ ]:
# === mcp_stdio_helper.py (또는 노트북 첫 셀) ===
import sys, os, asyncio
from pathlib import Path
from contextlib import asynccontextmanager

# 주피터 환경 권장: nest_asyncio
try:
    import nest_asyncio
    nest_asyncio.apply()
except Exception:
    pass

os.environ.setdefault("PYTHONUNBUFFERED", "1")  # 서브프로세스 버퍼링 방지

from mcp.client.stdio import stdio_client, StdioServerParameters
from mcp import ClientSession
from langchain_mcp_adapters.tools import load_mcp_tools

def _win_proactor_policy():
    if sys.platform == "win32":
        try:
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        except Exception:
            pass

@asynccontextmanager
async def start_stdio_mcp(
    server_file: str,
    *,
    python: str = sys.executable,
    cwd: str | None = None,
    env: dict | None = None,
    errlog_path: str | None = "mcp_stderr.log",
):
    """
    STDIO MCP 서버를 서브프로세스로 올리고 ClientSession과 변환된 LangChain tools를 반환합니다.
    사용 예:
        async with start_stdio_mcp("server_stdio.py") as (session, tools):
            ...
    """
    _win_proactor_policy()

    server_path = Path(server_file).resolve()
    if not server_path.exists():
        raise FileNotFoundError(f"서버 스크립트가 없습니다: {server_path}")

    params = StdioServerParameters(
        command=python,
        args=["-u", str(server_path)],            # -u: unbuffered
        cwd=str(cwd or server_path.parent),
        env=(env or os.environ).copy(),
    )

    err_fp = None
    try:
        if errlog_path:
            err_fp = open(errlog_path, "wb")
        async with stdio_client(params, errlog=err_fp) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await load_mcp_tools(session)
                try:
                    yield session, tools
                finally:
                    # 세션 종료는 with 블록 해제 시 자동
                    pass
    except Exception as e:
        # 진단을 돕기 위해 stderr tail 표시
        tail = ""
        try:
            if errlog_path and Path(errlog_path).exists():
                data = Path(errlog_path).read_bytes()
                tail = (data[-4096:] if len(data) > 4096 else data).decode("utf-8", "ignore")
        except Exception:
            pass
        raise RuntimeError(f"STDIO MCP 연결 실패: {e}\n\n=== server stderr (tail) ===\n{tail}") from e
    finally:
        if err_fp:
            err_fp.close()


In [ ]:
# 같은 디렉터리에 server_stdio.py 가 있어야 합니다.
# server_stdio.py 안에서는 반드시 로그 print 를 stderr 로 보내세요:
# print("MCP running...", file=sys.stderr)
results = await run_mcp_client_stdio_in_notebook(
    server_file="mcp_rag_stdio.py",
    user_prompt="DNA 서열 예측",
)

# for r in results:
#     print(r)


C:\Users\skyop\AppData\Local\Temp\ipykernel_2392\802431649.py:40: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(ChatOpenAI(model="gpt-4.1-mini", temperature=0), tools)


In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# 1) 서버 올리고 도구 받기
async with start_stdio_mcp("mcp_rag_stdio.py") as (session, tools):
    # 2) 에이전트 구성 & 실행
    model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
    agent = create_react_agent(model, tools)

    inputs = {"messages": [HumanMessage(content="jaeho", name="user")]}
    async for event in agent.astream(inputs):
        print(event)

    # # 한 번에 결과만:
    # result = await agent.ainvoke(inputs)
    # print(result)


C:\Users\skyop\AppData\Local\Temp\ipykernel_31800\2226912910.py:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model, tools)


{'agent': {'messages': [AIMessage(content='Hello! How can I assist you with "jaeho"? Are you looking for information about a person named Jaeho, or something else related to that name? Please provide more details.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 105, 'total_tokens': 145, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_4c2851f862', 'id': 'chatcmpl-CYEond9sfRw3SL6upBsDeSZn0GqPg', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--3c686554-cdcb-43a6-b01e-8d4d1065fc21-0', usage_metadata={'input_tokens': 105, 'output_tokens': 40, 'total_tokens': 145, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {

In [ ]:
# 같은 디렉터리에 server_stdio.py 가 있어야 합니다.
# server_stdio.py 안에서는 반드시 로그 print 를 stderr 로 보내세요:
# print("MCP running...", file=sys.stderr)
results = await run_mcp_client_stdio_in_notebook(
    server_file="mcp_rag_stdio.py",
    user_prompt="DNA 서열 예측",
)

# for r in results:
#     print(r)


In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-5-mini")

In [ ]:
import langchain
langchain.__version__

'1.0.3'

In [ ]:
import langchain_core
langchain_core.__version__

'1.0.3'